In [ ]:
import pandas as pd
import numpy as np
import re
from google.colab import files

In [ ]:
file_path = r"/content/sales.xlsx"
df = pd.read_excel(file_path)

In [ ]:
df

,No. Urut,Ref,Tgl Ref,No. Ref,No. Dok Ref,Kode Cust,Nama Cust,Alamat Cust,Kode Panel,Nama Panel,...,Hrg Unit Riil(Mu),Total Riil(Mu),Subtot Disc Riil(Mu),Subtot Riil(Mu),Disc Akh(Mu),Subtot Net(Mu),Pjk(Mu),Subtot Net Pjk(Mu),Kode Kota,Nama Kota
0,81.0,FJ,2025-05-02,2505-MX0005,2505-MX0005,C-CV--003,CV SUPER MART SENTOSA,URIMESING,NaN,NaN,...,27871.62,222972.97,0.0,222972.97,0,222972.97,24527.03,247500.0,AMBON,AMBON
1,82.0,FJ,2025-05-02,2505-MX0005,2505-MX0005,C-CV--003,CV SUPER MART SENTOSA,URIMESING,NaN,NaN,...,7700.00,23100.00,0.0,23100.00,0,23100.00,2541.00,25641.0,AMBON,AMBON
2,83.0,FJ,2025-05-02,2505-MX0005,2505-MX0005,C-CV--003,CV SUPER MART SENTOSA,URIMESING,NaN,NaN,...,7700.00,23100.00,0.0,23100.00,0,23100.00,2541.00,25641.0,AMBON,AMBON
3,84.0,FJ,2025-05-02,2505-MX0005,2505-MX0005,C-CV--003,CV SUPER MART SENTOSA,URIMESING,NaN,NaN,...,124324.32,124324.32,0.0,124324.32,0,124324.32,13675.68,138000.0,AMBON,AMBON
4,94.0,FJ,2025-05-02,2505-MX0004,2505-MX0004,CUST0066,SANDY,PASSO,NaN,NaN,...,92162.16,92162.16,0.0,92162.16,0,92162.16,10137.84,102300.0,AMBON,AMBON
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,147.0,FJ,2025-05-15,2505-MX0349,2505-MX0349,DAMAI-SEJAHTERA,CV. DAMAI SEJAHTERA,JL WOLTER MONGIN SIDI,NaN,NaN,...,124324.32,248648.65,0.0,248648.65,0,248648.65,27351.35,276000.0,AMBON,AMBON
201,148.0,FJ,2025-05-15,2505-MX0349,2505-MX0349,DAMAI-SEJAHTERA,CV. DAMAI SEJAHTERA,JL WOLTER MONGIN SIDI,NaN,NaN,...,210810.81,210810.81,0.0,210810.81,0,210810.81,23189.19,234000.0,AMBON,AMBON
202,149.0,FJ,2025-05-15,2505-MX0349,2505-MX0349,DAMAI-SEJAHTERA,CV. DAMAI SEJAHTERA,JL WOLTER MONGIN SIDI,NaN,NaN,...,124324.32,124324.32,0.0,124324.32,0,124324.32,13675.68,138000.0,AMBON,AMBON
203,150.0,FJ,2025-05-15,2505-MX0349,2505-MX0349,DAMAI-SEJAHTERA,CV. DAMAI SEJAHTERA,JL WOLTER MONGIN SIDI,NaN,NaN,...,15502.97,31005.95,0.0,31005.95,0,31005.95,3410.65,34416.6,AMBON,AMBON


# Data Cleaning

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 48 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   No. Urut              205 non-null    float64       
 1   Ref                   205 non-null    object        
 2   Tgl Ref               205 non-null    datetime64[ns]
 3   No. Ref               205 non-null    object        
 4   No. Dok Ref           205 non-null    object        
 5   Kode Cust             205 non-null    object        
 6   Nama Cust             205 non-null    object        
 7   Alamat Cust           205 non-null    object        
 8   Kode Panel            0 non-null      float64       
 9   Nama Panel            0 non-null      float64       
 10  Kode Sales            205 non-null    object        
 11  Nama Sales            205 non-null    object        
 12  Keterangan            30 non-null     object        
 13  Pajak               

## Cleansing column "No. Urut"

In [ ]:
df["No. Urut"] = df["No. Urut"].astype(str).str.replace(r"\.$", "", regex=True)
df["No. Urut"] = pd.to_numeric(df["No. Urut"], errors="coerce").astype("Int64")

## Cleansing column "Kode Sales"

In [ ]:
df[['Kode Sales', 'Nama Sales']].drop_duplicates().sort_values(['Kode Sales', 'Nama Sales'])

,Kode Sales,Nama Sales
31,EL01,Erwin Taneo
113,SL10,Erwin Taneo
10,SL10,FENDRY
4,SL12,Erwin Taneo
0,SL12,FENDRY


In [ ]:
# Because the sales code values are not unique and some codes are associated with more than one sales name, the sales codes need to be standardised for consistency.
# ensured each sales code uniquely maps to one salesperson.
df['Kode Sales'] = np.where(
    df['Nama Sales'] == 'Erwin Taneo', 'SL12',
    np.where(df['Nama Sales'] == 'FENDRY', 'SL10', df['Kode Sales']))
df[['Kode Sales', 'Nama Sales']].drop_duplicates().sort_values(['Kode Sales', 'Nama Sales'])

,Kode Sales,Nama Sales
0,SL10,FENDRY
4,SL12,Erwin Taneo


## Cleansing column "Kode Barang"

In [ ]:
df[['Kode Barang', 'Sat', 'Sat Std']].drop_duplicates().sort_values(['Kode Barang', 'Sat', 'Sat Std'])

,Kode Barang,Sat,Sat Std
56,4100000002,CRT,PCS
2,4100000002,PCS,PCS
153,4100000009,CRT,BTL
29,4100000047,BTL,BTL
164,4100000047,CRT,BTL
...,...,...,...
152,4100000964.GT,CRT,PCS
17,4100000964.GT,RCG,PCS
11,4100001079NEW,PCS,PCS
27,4100001080NEW,PCS,PCS


In [ ]:
# standardization "Kode Barang"
def clean_kode_barang(kode):
    if pd.isna(kode):
        return None
    kode_str = str(kode)
    kode_clean = re.sub(r'[^\d]', '', kode_str)
    return kode_clean

df['Kode Barang'] = df['Kode Barang'].apply(clean_kode_barang)
df[['Kode Barang', 'Sat', 'Sat Std']].drop_duplicates().sort_values(['Kode Barang', 'Sat', 'Sat Std'])

,Kode Barang,Sat,Sat Std
56,4100000002,CRT,PCS
2,4100000002,PCS,PCS
153,4100000009,CRT,BTL
29,4100000047,BTL,BTL
164,4100000047,CRT,BTL
...,...,...,...
136,4100001396,CRT,PCS
18,4100001396,RTG,PCS
161,4100001397,CRT,PCS
19,4100001397,RTG,PCS


# Download Clean Data

In [ ]:
df_sorted = df.sort_values(by="No. Urut", ascending=True).reset_index(drop=True)

In [ ]:
df_sorted

,No. Urut,Ref,Tgl Ref,No. Ref,No. Dok Ref,Kode Cust,Nama Cust,Alamat Cust,Kode Panel,Nama Panel,...,Hrg Unit Riil(Mu),Total Riil(Mu),Subtot Disc Riil(Mu),Subtot Riil(Mu),Disc Akh(Mu),Subtot Net(Mu),Pjk(Mu),Subtot Net Pjk(Mu),Kode Kota,Nama Kota
0,1,FJ,2025-05-05,2505-MX0065,2505-MX0065,110--KIO0541,RAFAEL,SETIA BUDI,NaN,NaN,...,92162.16,460810.81,0.0,460810.81,0,460810.81,50689.19,511500.00,AMBON,AMBON
1,2,FJ,2025-05-05,2505-MX0065,2505-MX0065,110--KIO0541,RAFAEL,SETIA BUDI,NaN,NaN,...,92162.16,460810.81,0.0,460810.81,0,460810.81,50689.19,511500.00,AMBON,AMBON
2,3,FJ,2025-05-13,2505-MX0279,2505-MX0279,104-CAMPINA-CAM0001,PINK TAWIRI,SAMPING J & J TAWIRI,NaN,NaN,...,92162.16,184324.32,0.0,184324.32,0,184324.32,20275.68,204600.00,AMBON,AMBON
3,4,FJ,2025-05-13,2505-MX0279,2505-MX0279,104-CAMPINA-CAM0001,PINK TAWIRI,SAMPING J & J TAWIRI,NaN,NaN,...,92162.16,92162.16,0.0,92162.16,0,92162.16,10137.84,102300.00,AMBON,AMBON
4,5,FJ,2025-05-07,2505-MX0115,2505-MX0115,110--ADE0624,ADE LALI PASSO,DEPAN LAP. PERIKANAN,NaN,NaN,...,15315.32,15315.32,0.0,15315.32,0,15315.32,1684.68,17000.00,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,201,FJ,2025-05-08,2505-MX0156,2505-MX0156,YON--0001,WALE,BENTAS,NaN,NaN,...,92162.16,92162.16,0.0,92162.16,0,92162.16,10137.84,102300.00,AMBON,AMBON
201,202,FJ,2025-05-08,2505-MX0156,2505-MX0156,YON--0001,WALE,BENTAS,NaN,NaN,...,92162.16,92162.16,0.0,92162.16,0,92162.16,10137.84,102300.00,AMBON,AMBON
202,203,FJ,2025-05-08,2505-MX0156,2505-MX0156,YON--0001,WALE,BENTAS,NaN,NaN,...,81081.08,81081.08,0.0,81081.08,0,81081.08,8918.92,90000.00,AMBON,AMBON
203,204,FJ,2025-05-08,2505-MX0156,2505-MX0156,YON--0001,WALE,BENTAS,NaN,NaN,...,90090.16,90090.16,0.0,90090.16,0,90090.16,9909.92,100000.08,AMBON,AMBON


In [ ]:
df_sorted.to_csv("sales_clean.csv", index=False)
files.download('sales_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>